In [1]:
import os, sys
project_root = os.path.abspath('..').replace('\\', '/')
if project_root not in [p.replace('\\', '/') for p in sys.path]:
    sys.path.append(project_root)

# 08 规则引擎模块 (core.rules)

提供规则定义、评估和报告功能。

**数据说明**: 基于 `hscredit_yyp.xlsx`

In [2]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from hscredit import init_setting
from hscredit.core.rules import Rule, get_columns_from_query, optimize_expr, beautify_expr

init_setting()

# 加载数据
_roots = [Path.cwd(), Path.cwd() / 'examples', Path.cwd().parent]
_fp = None
for _r in _roots:
    for _n in ('hscredit_yyp.xlsx', 'hengshucredit_yyp.xlsx'):
        if (_r / _n).is_file():
            _fp = _r / _n
            break
    if _fp is not None:
        break
if _fp is None:
    raise FileNotFoundError('请将 hscredit_yyp.xlsx 放在 examples/')

df = pd.read_excel(_fp)

# 构造目标变量
df['target'] = (df['MOB1'] > 3).astype(int)

# 重命名列为英文（pandas query 语法需要）
df = df.rename(columns={
    '中智小牛分C3': 'score_c3',
    '珊瑚92': 'score_coral',
    '极光欺诈分6v1': 'score_fraud',
    '青云24': 'score_qingyun',
    '占信V3': 'score_zhanxin'
})

print(f"样本数: {len(df):,}")
print(f"坏样本率: {df['target'].mean():.2%}")

样本数: 970
坏样本率: 16.70%


## 1. 基础规则定义

In [3]:
# 创建规则
rule1 = Rule(
    expr="score_c3 < 500 and score_coral > 80",
    name="高风险规则",
    description="信用分低且风险分高",
    weight=100
)

print(f"规则名称: {rule1.name}")
print(f"规则表达式: {rule1.expr}")
print(f"规则描述: {rule1.description}")
print(f"规则权重: {rule1.weight}")

# 从表达式中提取字段
columns = get_columns_from_query(rule1.expr)
print(f"\n涉及字段: {columns}")

规则名称: 高风险规则
规则表达式: score_c3 < 500 and score_coral > 80
规则描述: 信用分低且风险分高
规则权重: 100

涉及字段: ['score_c3', 'score_coral']


In [4]:
# 评估规则 - Rule 类直接使用 predict，无需 fit
result = rule1.predict(df)

print(f"命中规则样本数: {result.sum():,}")
print(f"规则命中率: {result.mean():.2%}")

# 查看命中样本的坏账率
hit_bad_rate = df[result]['target'].mean()
overall_bad_rate = df['target'].mean()
print(f"命中样本坏账率: {hit_bad_rate:.2%}")
print(f"总体坏账率: {overall_bad_rate:.2%}")
print(f"Lift: {hit_bad_rate/overall_bad_rate:.2f}")

命中规则样本数: 15
规则命中率: 1.55%
命中样本坏账率: 40.00%
总体坏账率: 16.70%
Lift: 2.40


In [5]:
# 优化和美化表达式
complex_expr = "((score_c3>=500) and (score_c3<=650)) or ((score_coral<500)and(score_fraud>0.6))"

print("原始表达式:")
print(complex_expr)

# 美化表达式
beautified = beautify_expr(complex_expr)
print("\n美化后表达式:")
print(beautified)

# 优化表达式
optimized = optimize_expr(complex_expr)
print("\n优化后表达式:")
print(optimized)

原始表达式:
((score_c3>=500) and (score_c3<=650)) or ((score_coral<500)and(score_fraud>0.6))

美化后表达式:
(score_c3 >= 500 & score_c3 <= 650) | (score_coral < 500 & score_fraud > 0.6)

优化后表达式:
(score_c3 >= 500 & score_c3 <= 650) | (score_coral < 500 & score_fraud > 0.6)


## 2. 规则组合

In [8]:
# 创建多个规则
rule_age = Rule(
    expr="score_c3 < 550",
    name="低分规则",
    weight=50
)

rule_risk = Rule(
    expr="score_coral > 80",
    name="高风险规则",
    weight=50
)

# AND 组合
combined_and = rule_age & rule_risk
result_and = combined_and.predict(df)
print(f"AND组合规则命中: {result_and.sum()} 条")

# OR 组合
combined_or = rule_age | rule_risk
result_or = combined_or.predict(df)
print(f"OR组合规则命中: {result_or.sum()} 条")

AND组合规则命中: 55 条
OR组合规则命中: 270 条


## 3. 规则有效性分析

In [7]:
# 分析多个规则的效果
rules = [
    Rule("score_c3 < 500", name="极低分规则", weight=100),
    Rule("score_coral > 85", name="极高风险规则", weight=80),
    Rule("score_fraud > 0.8", name="欺诈风险规则", weight=90)
]

results = []
for rule in rules:
    mask = rule.predict(df)
    hit_count = mask.sum()
    hit_rate = mask.mean()
    hit_bad_rate = df[mask]['target'].mean() if hit_count > 0 else 0
    overall_bad_rate = df['target'].mean()
    lift = hit_bad_rate / overall_bad_rate if overall_bad_rate > 0 else 0
    results.append({
        '规则名': rule.name,
        '命中率': f"{hit_rate:.2%}",
        '命中坏账率': f"{hit_bad_rate:.2%}",
        'Lift': f"{lift:.2f}",
        '权重': rule.weight
    })

results_df = pd.DataFrame(results)
print("=== 规则有效性分析 ===")
display(results_df)

=== 规则有效性分析 ===


,规则名,命中率,命中坏账率,Lift,权重
0,极低分规则,1.75%,35.29%,2.11,100
1,极高风险规则,27.22%,14.02%,0.84,80
2,欺诈风险规则,0.82%,37.50%,2.25,90
